# 01 — Ingestão e Bronze (exercício)

**Objetivo:** praticar ingestão via HuggingFace, normalização de JSON e escrita de Parquet.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (camada **bronze**):

> Fonte → **Bronze** → Staging → Dim/Fact → Checks → Profiling → Marts/Views → Dashboard

Aqui você baixa o dataset do HuggingFace, normaliza campos aninhados em `*_json` e grava `data/bronze_hackerone_reports.parquet`. Esse arquivo é a entrada da próxima etapa.

In [7]:
# Parâmetros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    if (candidato / "_common.py").exists():
        sys.path.insert(0, str(candidato))
        break
    filho = candidato / "sessao-02-data-architecture"
    if (filho / "_common.py").exists():
        sys.path.insert(0, str(filho))
        break

# O erro 'ModuleNotFoundError: No module named 'exercicios'' ocorre porque
# '_common.py' foi encontrado diretamente no sys.path, não dentro de um pacote 'exercicios'.
# A importação deve ser feita diretamente do módulo '_common'.
# from exercicios._common import configurar_paths, conectar_duckdb # Linha original com erro
import _common
paths = _common.configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")

Raiz do projeto: /content
Banco DuckDB: /content/data/hackerone.duckdb


## Exercício
Complete os TODOs para carregar os três splits, normalizar campos aninhados e gravar a camada bronze.

In [10]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json

HUGGINGFACE_DATASET = "Hacker0x01/disclosed_reports"

HUGGINGFACE_DATASET = "Hacker0x01/disclosed_reports"

train_ds = load_dataset(HUGGINGFACE_DATASET, split="train").to_pandas()
test_ds = load_dataset(HUGGINGFACE_DATASET, split="test").to_pandas()
validate_ds = load_dataset(HUGGINGFACE_DATASET, split="validation").to_pandas()

df = pd.concat([train_ds, test_ds, validate_ds], ignore_index=True)
print(f"Linhas carregadas: {len(df):,}")
df.head(3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.87M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10094 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1262 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1262 [00:00<?, ? examples/s]

Linhas carregadas: 12,618


,id,title,created_at,substate,vulnerability_information,reporter,team,has_bounty?,visibility,disclosed_at,weakness,original_report_id,vote_count,structured_scope
0,411337,Forget password link not expiring after email ...,2018-09-19T05:13:33.396Z,resolved,I found a token miss configuration flaw in cha...,"{'cleared': True, 'disabled': False, 'hacker_m...","{'awards_miles': False, 'default_currency': 'u...",True,full,2018-09-20T06:42:43.088Z,"{'id': 124.0, 'name': 'Improper Authorization'}",NaN,35,"{'asset_identifier': 'chaturbate.com', 'asset_..."
1,311805,Cross-origin resource sharing misconfig,2018-02-02T21:19:34.071Z,duplicate,Description\nAn HTML5 cross-origin resource sh...,"{'cleared': False, 'disabled': False, 'hacker_...","{'awards_miles': False, 'default_currency': 'u...",False,full,2018-03-13T14:23:52.379Z,"{'id': 27.0, 'name': 'Improper Authentication ...",193559.0,3,None
2,385322,Open API For Username enumeration,2018-07-23T07:32:50.020Z,not-applicable,"We Can do username enumeration,\nReproduce:\n1...","{'cleared': True, 'disabled': False, 'hacker_m...","{'awards_miles': False, 'default_currency': 'u...",False,full,2018-07-23T14:33:47.328Z,None,NaN,24,None


In [12]:
bronze_cols = [
    "id", "title", "created_at", "disclosed_at", "substate", "visibility", "has_bounty?",
    "vote_count", "original_report_id", "reporter_json", "team_json", "weakness_json",
    "structured_scope_json", "vulnerability_information",
]
bronze = df[[c for c in bronze_cols if c in df.columns]].copy()
bronze.to_parquet(BRONZE_PARQUET, index=False)
bronze.to_csv(RAW_CSV, index=False)

print(f"Bronze Parquet gravado em: {BRONZE_PARQUET}")
print(f"CSV bruto gravado em: {RAW_CSV}")
print(f"Shape bronze: {bronze.shape}")
bronze.head(3)

Bronze Parquet gravado em: /content/data/bronze_hackerone_reports.parquet
CSV bruto gravado em: /content/data/raw_data.csv
Shape bronze: (12618, 14)


,id,title,created_at,disclosed_at,substate,visibility,has_bounty?,vote_count,original_report_id,reporter_json,team_json,weakness_json,structured_scope_json,vulnerability_information
0,411337,Forget password link not expiring after email ...,2018-09-19 05:13:33.396000+00:00,2018-09-20 06:42:43.088000+00:00,resolved,full,True,35,NaN,"{""cleared"": true, ""disabled"": false, ""hacker_m...","{""awards_miles"": false, ""default_currency"": ""u...","{""id"": 124.0, ""name"": ""Improper Authorization""}","{""asset_identifier"": ""chaturbate.com"", ""asset_...",I found a token miss configuration flaw in cha...
1,311805,Cross-origin resource sharing misconfig,2018-02-02 21:19:34.071000+00:00,2018-03-13 14:23:52.379000+00:00,duplicate,full,False,3,193559.0,"{""cleared"": false, ""disabled"": false, ""hacker_...","{""awards_miles"": false, ""default_currency"": ""u...","{""id"": 27.0, ""name"": ""Improper Authentication ...",{},Description\nAn HTML5 cross-origin resource sh...
2,385322,Open API For Username enumeration,2018-07-23 07:32:50.020000+00:00,2018-07-23 14:33:47.328000+00:00,not-applicable,full,False,24,NaN,"{""cleared"": true, ""disabled"": false, ""hacker_m...","{""awards_miles"": false, ""default_currency"": ""u...",{},{},"We Can do username enumeration,\nReproduce:\n1..."


In [ ]:
bronze_cols = ["id", "title", "created_at", "disclosed_at", "substate", "visibility", "has_bounty?", "vote_count", "original_report_id", "reporter_json", "team_json", "weakness_json", "structured_scope_json", "vulnerability_information"]
# TODO 6: selecione apenas as colunas disponíveis em bronze_cols.
bronze = ...
# TODO 7: grave em BRONZE_PARQUET e RAW_CSV sem índice.
...
print(f"Shape bronze: {bronze.shape}")